# 05 — Huấn luyện ML-XC (JAX + Flax)
Mô hình học `ΔE_θ[n] = Σ w ρ f_θ(ρ, s)` cộng vào năng lượng PBE để khớp nhãn `Δ = E_CCSD(T) − E_PBE`.

- **Loss** = MSE của Δ trên mỗi nguyên tử (kcal/mol/nguyên tử) + phạt vi phạm ràng buộc Lieb–Oxford (`E_xc ≥ −2.273 ∫ n^{4/3}`, với E_xc = E_xc^PBE + ΔE_θ). Trên dữ liệu thật PBE cách xa biên này nên số phạt thường bằng 0.
- Tập train gồm các phân tử train **và 5 nguyên tử cô lập** (để học các độ lệch theo nguyên tố).
- **Validation** (phân tử val) dùng để dừng sớm và chọn mô hình theo RMSE năng lượng nguyên tử hóa. **Tập test không được nạp ở notebook này.**
- Dữ liệu chỉ có ~30 phân tử train nên lưới siêu tham số gồm cả cấu hình gốc (128,128,64) lẫn các mô hình nhỏ hơn có weight decay; validation quyết định.

**Đầu ra:** `data/checkpoints/<cấu hình>.msgpack` (+ `.json`), `best_model.msgpack`, `train_log.json`.

In [ ]:
# Cài thư viện (chỉ trên Colab; runtime Colab bị reset sau mỗi phiên). Chạy cục bộ: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip install -q flax optax h5py matplotlib

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import jax
from utils import data_utils as du
from utils import model_utils as mu

H5 = f"{PROC}/qm9_xc_dataset.h5"
print("JAX devices:", jax.devices())

EPOCHS, BATCH, PATIENCE, SEED, LAM_LO = 300, 8, 40, 0, 1.0
GRID = [
    {"name": "spec_128_128_64", "hidden": (128, 128, 64), "lr": 1e-4, "wd": 0.0},      # cấu hình trong README gốc
    {"name": "small_16_16",     "hidden": (16, 16),       "lr": 3e-3, "wd": 1e-4},
    {"name": "small_32_32",     "hidden": (32, 32),       "lr": 1e-3, "wd": 1e-4},
    {"name": "small_32_32_wd",  "hidden": (32, 32),       "lr": 3e-3, "wd": 1e-3},
]

## Dữ liệu (chỉ train và val)

In [ ]:
tab_all = du.energy_table(H5)
table = tab_all[tab_all.split != "test"].copy()          # loại hẳn test khỏi mọi thứ ở notebook này
assert "test" not in set(table.split)
atoms = [n for n in table.index if table.loc[n, "is_atom"]]
tr_mols = [n for n in table.index if table.loc[n, "split"] == "train" and not table.loc[n, "is_atom"]]
va_mols = [n for n in table.index if table.loc[n, "split"] == "val"]
print(f"train: {len(tr_mols)} phân tử + {len(atoms)} nguyên tử | val: {len(va_mols)} phân tử")

load = lambda names: mu.make_dataset([du.load_system(H5, n) for n in names], du.reduced_gradient)
atom_ds, tr_ds, va_ds = load(atoms), load(tr_mols), load(va_mols)
train_ds = tr_ds + atom_ds
print("Số điểm lưới (đã đệm) lớn nhất:", max(len(d["rho"]) for d in train_ds + va_ds))

pbe_val = du.error_stats(du.ae_error_kcal(table, va_mols, "e_pbe"))
print("Baseline PBE trên val (sai số năng lượng nguyên tử hóa, kcal/mol):", {k: round(v, 2) for k, v in pbe_val.items()})

In [ ]:
def make_eval(val_names, val_ds):
    def eval_fn(params, de_fn):
        dE = mu.predict_delta(de_fn, params, val_ds + atom_ds)
        err = du.ae_error_kcal(table, val_names, "e_pbe", dE)
        return float(np.sqrt(np.mean(err ** 2))), {}
    return eval_fn

results = []
for g in GRID:
    print(f"\n=== {g['name']}: hidden={g['hidden']} lr={g['lr']} wd={g['wd']} ===")
    t0 = time.time()
    net = mu.XCNet(hidden=g["hidden"])
    params, log, de_fn = mu.train(net, train_ds, make_eval(va_mols, va_ds), lr=g["lr"], weight_decay=g["wd"],
                                  epochs=EPOCHS, batch_size=BATCH, patience=PATIENCE, lam_lo=LAM_LO, seed=SEED)
    log.update(name=g["name"])
    mu.save_checkpoint(f"{CKPT}/{g['name']}.msgpack", params, {**log, "history": None})
    results.append({"cfg": g, "params": params, "log": log})
    print(f"val RMSE (AE) tốt nhất: {log['best_val']:.3f} kcal/mol | {log['epochs_run']} epoch | {time.time()-t0:.0f}s")

In [ ]:
best = min(results, key=lambda r: r["log"]["best_val"])
mu.save_checkpoint(f"{CKPT}/best_model.msgpack", best["params"], {**best["log"], "history": None})
json.dump([{"name": r["cfg"]["name"], "hidden": list(r["cfg"]["hidden"]), "lr": r["cfg"]["lr"], "wd": r["cfg"]["wd"],
            "best_val_rmse_AE_kcal": r["log"]["best_val"], "epochs_run": r["log"]["epochs_run"],
            "history": r["log"]["history"]} for r in results], open(f"{CKPT}/train_log.json", "w"), indent=1, default=float)
print("Chọn:", best["cfg"]["name"], "| val RMSE =", round(best["log"]["best_val"], 3), "kcal/mol (PBE: %.3f)" % pbe_val["rmse"])
if best["log"]["best_val"] >= pbe_val["rmse"]:
    print("LƯU Ý: mô hình chưa tốt hơn PBE trên validation.")

plt.figure(figsize=(6.5, 4))
for r in results:
    h = pd.DataFrame(r["log"]["history"])
    plt.plot(h.epoch, h.val, label=r["cfg"]["name"])
plt.axhline(pbe_val["rmse"], color="k", ls="--", label="PBE")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("val RMSE năng lượng nguyên tử hóa (kcal/mol)")
plt.legend(); plt.tight_layout(); plt.savefig(f"{CKPT}/val_curves.png", dpi=120); plt.show()